In [ ]:
# make local editable packages automatically reload
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Qualitative Evaluation cellposeSAM

## 1. Import Libraries

In [ ]:
import os
import logging
from cellpose.models import CellposeModel
import numpy as np
from cellpose import core, io, plot
import matplotlib.pyplot as plt

plt.style.use('default')

logger = io.logger_setup() # run this to get printing of progress
logging.getLogger().setLevel(logging.INFO)

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model_path = "/home/wangchuyao/.cellpose/models/cpsam"

# Check if the model file exists before attempting to load
if os.path.exists(model_path):
    # Initialize the CellposeModel with your local model file
    # You can also specify 'gpu=True' if you want to use a GPU, and 'device' for a specific device.
    model = CellposeModel(pretrained_model=model_path, gpu=True)
    print(f"Successfully loaded model from: {model_path}")
else:
    print(f"Error: Model file not found at {model_path}")

## 2. Import files

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern, find_dapi_channel_file
from image_processing_tools.image_class.image_container import ImageContainer
import logging

# Set up basic logging
logging.basicConfig(level=logging.INFO) # change to DEBUG for debugging information

config = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1024,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

search_path = (
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV1_1_CY5, CY3.5 NAR, FITC, DAPI, BF",
    "~/A8/Data_Roan/250704_CA_Ca9_WT_smFISH_Cocu_SSU_GAPDH/CA_SSU_GAPDH_Infection_FOV2_1_CY5, CY3.5 NAR, FITC, DAPI",
    "~/A8/Data_Roan/250925_Cocu_Dyes_cFOS_CSF3_ECE1/Cocu_Cellbrite_cFOS_CSF3_ECE1_01_CY5, CY3.5 NAR, CY3, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI",
    "~/A8/Data_Roan/251114_MonoCa9_Phalloidin/CA9_Phalloidin_01_CY5, FITC, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_DIC",
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[5], file_pattern[0], verbose=True)
# files = files + find_files_by_pattern(search_path[6], file_pattern[0], verbose=True)

# --- Create the two-channel composite image ---

# TODO: For new datasets check DAPI path is correct
DAPI_INDEX = find_dapi_channel_file(files)
num_channels = len(files)

if num_channels > 1 and 0 <= DAPI_INDEX < num_channels:
    # The DAPI (nucleus) channel is at the specified index
    dapi_channel = files[DAPI_INDEX]
    
    # All other channels are grouped as cytoplasmic stains
    cytoplasmic_channels = [file for i, file in enumerate(files) if i != DAPI_INDEX]

    # Create a nested list to define the composition:
    # - The first element is a list of channel paths to be summed.
    # - The second element is the DAPI channel path.
    composition_structure = [cytoplasmic_channels, dapi_channel]

    # Instantiate the CompositeImage with this structure
    composite_image = ImageContainer(composition_structure, config)

    # The merge() method will now produce a 2-channel image
    two_channel_output = composite_image.merge()

    print(f"\nSuccessfully created a two-channel composite image.")
    print(f"Output image shape: {two_channel_output.shape}")
    print(f"Output image data type: {two_channel_output.dtype}")

    # You can now use 'two_channel_output' for further processing or visualization
else:
    print(f"Error: Cannot create composite. Found {num_channels} channels, but DAPI_INDEX is {DAPI_INDEX}.")

In [ ]:
composite_image.display(title='Combined')

## 3. Run Cellpose-SAM

### On cytoplasm + DAPI input

In [ ]:
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = 'None' # @param ['None', 0, 1, 2, 3, 4, 5]


selected_channels = []
for i, c in enumerate([first_channel, second_channel, third_channel]):
  if c == 'None':
    continue
  if int(c) > two_channel_output.shape[-1]:
    assert False, 'invalid channel index, must have index greater or equal to the number of channels'
  if c != 'None':
    selected_channels.append(int(c))

selected_channels = [0,1]
img_selected_channels = np.zeros_like(two_channel_output)
img_selected_channels[:, :, :len(selected_channels)] = two_channel_output[:, :, selected_channels]

flow_threshold = 1
cellprob_threshold = -10
tile_norm_blocksize = 0

masks, flows, styles = model.eval(img_selected_channels, 
                                  batch_size=32, 
                                  flow_threshold=flow_threshold, 
                                  cellprob_threshold=cellprob_threshold,
                                  normalize={"tile_norm_blocksize": tile_norm_blocksize})

fig = plt.figure(figsize=(12,5))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Assume 'cell_probability_map' and 'cellprob_threshold' are available ---
cell_probability_map = flows[2]
cellprob_threshold = 0.8

# --- Code to display the maps with a color bar ---

# 1. Create a figure with two subplots (1 row, 2 columns).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- Panel 1: Original Cell Probability Map ---

# 2. Display the probability map on the first axis.
#    'viridis' is a perceptually uniform colormap, great for this kind of data.
im = ax1.imshow(cell_probability_map, cmap='viridis')

# 3. Add a color bar and link it to the first plot.
cbar = fig.colorbar(im, ax=ax1, shrink=0.8)
cbar.set_label('Cell Probability', rotation=270, labelpad=15)

# 4. Add a title and turn off the axes for a cleaner look.
ax1.set_title("Cell Probability Map")
ax1.axis('off')


# --- Panel 2: Thresholded Cell Probability Map ---

# 5. Create the binary mask by applying the threshold.
thresholded_map = cell_probability_map > cellprob_threshold

# 6. Display the binary mask on the second axis.
ax2.imshow(thresholded_map, cmap='gray')

# 7. Add a title and turn off the axes.
ax2.set_title(f"Thresholded Map (>{cellprob_threshold})")
ax2.axis('off')


# 8. Adjust layout and show the final plot.
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from matplotlib.colors import LogNorm
from matplotlib.colorbar import ColorbarBase

def display_flow_magnitude_with_colorbar(rgb_flow_image: np.ndarray, title: str = "Flow Field Magnitude (Log Scale)"):
    """
    Displays the magnitude of an RGB flow field with a color bar.

    The magnitude is shown on a logarithmic scale to enhance the visibility of
    low-magnitude vectors in the dark regions of the flow visualization.

    Args:
        rgb_flow_image (np.ndarray): The RGB visualization of the flow field (e.g., flows[0]).
        title (str): The title for the plot.
    """
    if rgb_flow_image.ndim != 3 or rgb_flow_image.shape[2] != 3:
        raise ValueError("Input must be an RGB image with shape (H, W, 3).")

    # 1. Convert the RGB flow image to grayscale. Brightness corresponds to magnitude.
    # The values are typically in the [0, 255] range.
    magnitude = cv2.cvtColor(rgb_flow_image, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # Add a small epsilon to avoid log(0) issues.
    # magnitude += 1e-6

    # 2. Set up the plot with space for a color bar.
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))

    # 3. Display the magnitude image using a logarithmic norm.
    # This stretches the colormap in the low-value (dark) regions.
    # We set the range from our epsilon to the max value (255).
    # im = ax.imshow(magnitude, cmap='viridis', norm=LogNorm(vmin=1e-6, vmax=255.0))
    im = ax.imshow(magnitude, cmap='viridis')
    ax.set_title(title)
    ax.axis('off')

    # 4. Create a color bar that accurately reflects the logarithmic scale.
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Flow Vector Magnitude")

    plt.show()

display_flow_magnitude_with_colorbar(flows[0], title="Predicted Flow Field")


#### Channel inversion

In [ ]:
selected_channels = [1,0]
img_selected_channels = np.zeros_like(two_channel_output)
img_selected_channels[:, :, :len(selected_channels)] = two_channel_output[:, :, selected_channels]

flow_threshold = 0.8
cellprob_threshold = -3
tile_norm_blocksize = 0

masks, flows, styles = model.eval(img_selected_channels, 
                                  batch_size=32, 
                                  flow_threshold=flow_threshold, 
                                  cellprob_threshold=cellprob_threshold,
                                  normalize={"tile_norm_blocksize": tile_norm_blocksize})

fig = plt.figure(figsize=(12,5))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()

### Compare with 2 channels

In [ ]:
separate_channels = [ImageContainer(f,config) for f in files]

composite_images = []
for i, ch in enumerate(separate_channels):
    if i == DAPI_INDEX:
        continue # Skip the DAPI channel itself

    # Use the overloaded '+' operator to create a CompositeImage
    # This will create a CompositeImage containing the DAPI channel and the current 'other_channel_obj'.
    composite = separate_channels[i] + separate_channels[DAPI_INDEX]
    composite_images.append(composite)
    print(f"Created composite image: {composite.name}")

print(f"\nSuccessfully created {len(composite_images)} composite images.")

In [ ]:
import matplotlib.pyplot as plt
three_comp = ImageContainer(separate_channels,config)
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

separate_channels[0].display(ax=axes[0,0],title="CY5 (Phalloidin)")
separate_channels[1].display(ax=axes[0,1],title="DAPI")
separate_channels[2].display(ax=axes[1,0],title="FITC")
three_comp.display(ax=axes[1,1],title='Combined')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Define your colors and create a sample image ---

# Define the colormap as a (3, 3) array for your 3 channels.
# Each row is an RGB color for one channel.
colormap = np.array([
    [1.0, 1.0, 1.0],                  # Channel 0 -> White
    [0.992, 0.886, 0.208],            # Channel 1 -> Sunny Yellow (#FDE235)
    [0.600, 0.396, 0.082],             # Channel 2 -> Golden Brown (#996515)
])

# Create a sample H x W x 3 image to demonstrate the effect.
# Each channel has a gradient in a different part of the image.
image = three_comp.merge()

# --- 2. Create the colorized RGB image ---

# Use broadcasting to apply the colormap. This is a fast equivalent to a loop.
# This multiplies each channel of the input image by its corresponding color.
# image shape: (H, W, C) -> (H, W, C, 1)
# colormap shape: (C, 3) -> (1, 1, C, 3)
# Result of multiplication is (H, W, C, 3). We then sum over the channel axis.
colorized_image = (image[..., np.newaxis] * colormap[np.newaxis, np.newaxis, :, :]).sum(axis=2)

# --- 3. Rescale the result to the [0, 1] range for display ---
# This is the crucial step to prevent the image from being washed out or clipped.
min_val = colorized_image.min()
max_val = colorized_image.max()
if max_val > min_val:
    colorized_image = (colorized_image - min_val) / (max_val - min_val)

# Ensure values are clipped to the valid [0, 1] range.
colorized_image = np.clip(colorized_image, 0, 1)

# --- 4. Display the final image using imshow ---

fig, ax = plt.subplots(figsize=(4, 4))

fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.imshow(colorized_image)
ax.set_title("Colorized Image (Sum and Rescale Method)")
ax.axis('off')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

composite_images[0].display(ax=ax1,title="DAPI+CY5")
composite_images[1].display(ax=ax2,title="DAPI+FITC")

In [ ]:
for img in separate_channels:
    input = img.merge()
    masks, flows, styles = model.eval(input, 
                                    batch_size=32, 
                                    flow_threshold=0.4, 
                                    cellprob_threshold=0,
                                    normalize={"tile_norm_blocksize": 0})

    fig = plt.figure(figsize=(12,5))
    plot.show_segmentation(fig, input, masks, flows[0])
    plt.tight_layout()
    plt.show()

## 4. Run Custom Cellpose-SAM

In [ ]:
# Import your new custom class
from custom_cellpose_model import CellposeModelCustomMap
import numpy as np
import matplotlib.pyplot as plt
from cellpose import plot

selected_channels = [0,1]
img_selected_channels = np.zeros_like(two_channel_output)
img_selected_channels[:, :, :len(selected_channels)] = two_channel_output[:, :, selected_channels]

# 1. Initialize your custom model instead of the standard one
#    It will load the same pretrained weights.
model_path = "/home/wangchuyao/.cellpose/models/cpsam"
model = CellposeModelCustomMap(pretrained_model=model_path, gpu=True)

# 2. Run evaluation
# You can now pass your custom 'flow_field_threshold' directly to eval().
# The standard 'cellprob_threshold' is also passed to your custom function.
flow_threshold = 0.4
cellprob_threshold = 1
tile_norm_blocksize = 0
flow_field_threshold = 40

masks, flows, styles = model.eval(img_selected_channels, 
                                  batch_size=32, 
                                  flow_threshold=flow_threshold, 
                                  cellprob_threshold=cellprob_threshold,
                                  flow_field_threshold=flow_field_threshold,
                                  normalize={"tile_norm_blocksize": tile_norm_blocksize})

fig = plt.figure(figsize=(12,5))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()

In [ ]:
from custom_cellpose_model import combine_thresholded_maps

cell_probability_map = flows[2]
flow_field = flows[0]

map = combine_thresholded_maps(cell_prob_map=cell_probability_map,
                               flow_field=flow_field,
                               prob_threshold=cellprob_threshold,
                               flow_threshold=flow_field_threshold)

plt.imshow(map,cmap='gray')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.imshow(cell_probability_map>1,cmap='gray')
plt.axis('off')
plt.title('Cell Prob > 1')
plt.tight_layout()
plt.show()